# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [16]:
import sqlite3
import json
from openai import OpenAI
import gradio as gr

In [17]:
# Initialization


# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
MODEL = "llama3.2:latest"
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [18]:
DB = "prices.db"

# Ensure the database table exists with a UNIQUE constraint on city
def init_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS prices (
                city TEXT PRIMARY KEY,
                price REAL
            )
        ''')
        conn.commit()



In [19]:
init_db()

In [20]:
def set_ticket_price(destination_city, price, currency="USD"):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', 
            (destination_city.lower(), price, price)
        )
        conn.commit()
    return f"Successfully set price for {destination_city} to {price} {currency}."

def get_ticket_price(destination_city):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {destination_city} is ${result[0]}" if result else f"No price data available for {destination_city}."

In [21]:
# Tool Schemas
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    }
}

set_price_fn = {
    "name": "set_ticket_price",
    "description": "Set or update the ticket price for a specific destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The target destination city.",
            },
            "price": {
                "type": "number",
                "description": "The new price for the ticket.",
            },
            "currency": {
                "type": "string",
                "description": "The 3-letter currency code (e.g., USD, EUR).",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False,
    },
}

In [22]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_fn},
]

In [23]:
TOOL_REGISTRY = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price,
}

In [24]:


def handle_tool_calls(message):
    tool_calls = getattr(message, "tool_calls", None)
    if not tool_calls:
        return []

    responses = []
    for call in tool_calls:
        fn_name = call.function.name
        args = call.function.arguments
        
        # Ensure args is a dictionary
        if isinstance(args, str):
            args = json.loads(args)

        # Execute tool or fall back to an unknown tool response
        target_fn = TOOL_REGISTRY.get(fn_name)
        if target_fn:
            result = target_fn(**args)
            content = result if isinstance(result, str) else json.dumps(result)
        else:
            content = json.dumps({"error": f"Tool '{fn_name}' not supported."})

        responses.append({
            "role": "tool",
            "content": content,
            "tool_call_id": call.id,
        })

    return responses

In [28]:
system_message = """
You are a helpful assistant for an Airline called FlightAI, who has tone of someone who drank helium.
Give short, courteous answers, no more than 1 sentence, but maintain the tone.
Always be accurate. If you don't know the answer, say so.

CRITICAL INSTRUCTIONS:
1. NEVER output raw JSON or tool names in your spoken text.
2. Only call tools using official tool calls.
3. Only use the exact parameters specified in the tool schemas (e.g., destination_city). Do not invent parameters like dates or flight numbers.
4. If you need information, execute the tool silently without outputting JSON text to the user.
"""

In [29]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [30]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
